# HomeReach Melbourne — 03: Explore the rental data

**Stage:** first real-data exploratory data analysis (EDA). **Source snapshot:** September 2025.
**Scope:** 31 LGAs in the source's metropolitan groups; `2 bedroom flats`; 1999Q2–2025Q3.

We will answer three descriptive questions:
1. How do published median weekly rents compare across LGAs in September 2025?
2. How have the published medians changed over time in three selected LGAs?
3. How much did each LGA's median change from the same quarter one year earlier?

A fourth chart shows the published observation counts alongside the rents, to help us read those comparisons carefully.

**Use the existing `homereach` kernel.** Run the code cells from top to bottom with the cell's ▶ or **Ctrl+Enter**, not Code Runner. No new package installation is needed. This notebook reads the pipeline's **CSV copy**, using the existing `read_panel_csv()` helper to restore types. Your pipeline has already checked that CSV and Parquet contain the same data.

**We do not rebuild or overwrite the cleaned data.** Outputs go only under `data/interim/rent_eda/` and to the generated `docs/rental_eda_summary.md`. No registry, Git, dashboard, model, or network changes occur.

### Interpretation boundary
These are nominal (not inflation-adjusted) published area-level medians, not live listings, household affordability measures, individual-property price histories, or recommendations. Our changes in medians are **not** the publisher's composition-adjusted Rental Indices.

Publisher's definitions: https://www.dffh.vic.gov.au/publications/rental-report

## 1. Imports and project paths

We reuse the CSV-loading function from the module you just tested. This is why moving notebook logic into a module was useful: later analyses can import it rather than reimplementing the parser.

The notebook finds the project root whether VS Code starts in the main folder or in `notebooks/`.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys

import numpy as np
import pandas as pd
import plotly
import plotly.graph_objects as go
from IPython.display import display
from homereach.io.dffh_rent import (
    COLUMN_ORDER, KEY_COLUMNS, AGGREGATE_LABELS,
    read_panel_csv, load_snapshot, fingerprint, require,
)

pd.set_option("display.max_columns", 14)
pd.set_option("display.max_rows", 40)
pd.set_option("display.max_colwidth", 60)

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Open this notebook within the HomeReach project folder.")

ROOT = find_project_root(Path.cwd())
INPUT = ROOT / "data/processed/rent_quarterly.csv"
SPEC_PATH = ROOT / "config/rent_snapshot.json"
QUALITY_PATH = ROOT / "data/processed/rent_quality_report.json"
for required_path in [INPUT, SPEC_PATH, QUALITY_PATH]:
    if not required_path.is_file():
        raise FileNotFoundError(
            f"Missing {required_path.relative_to(ROOT)}. "
            "Complete the rental pipeline before this EDA; do not invent replacement data."
        )

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Plotly:", plotly.__version__)
print("Project root:", ROOT)
print("EDA input:", INPUT.relative_to(ROOT))

Python: c:\Users\PhamDang\anaconda3\envs\homereach\python.exe
pandas: 3.0.3
Plotly: 7.1.0
Project root: D:\My project\homereach_melbourne
EDA input: data\processed\rent_quarterly.csv


## 2. Load and verify the existing pipeline output

We check the CSV's fingerprint against the pipeline's quality report, then verify the schema, source metadata and quarterly coverage. Each LGA must have one row per quarter before we calculate a four-row lag.

The fingerprint proves correspondence to the locally recorded pipeline output; it is not an independent audit of the publisher. The earlier pipeline performed the source-cell checks. We do not repeat Excel ingestion here.

In [4]:
spec = load_snapshot(SPEC_PATH)
quality_report = json.loads(QUALITY_PATH.read_text(encoding="utf-8"))
input_sha256 = fingerprint(INPUT)
require(quality_report.get("source", {}).get("sha256") == spec["sha256"],
        "The quality report belongs to a different source snapshot.")
require(quality_report.get("output_sha256", {}).get(INPUT.name) == input_sha256,
        "The CSV does not match the pipeline quality report. Stop and review the files.")

panel = read_panel_csv(INPUT)
require(panel.columns.tolist() == COLUMN_ORDER, "Expected the validated 14-column schema.")
require(panel[KEY_COLUMNS].notna().all().all(), "Missing analytical key.")
require(not panel.duplicated(KEY_COLUMNS).any(), "Duplicate analytical keys.")
require(not panel["rental_area_name"].isin(AGGREGATE_LABELS).any(), "Aggregate row in LGA panel.")
for column, expected in {
    "area_type": "LGA", "property_type": spec["property_type"],
    "source_file": spec["filename"], "source_sheet": spec["sheet"],
    "source_reporting_quarter": spec["reporting_quarter"],
}.items():
    require(panel[column].notna().all() and panel[column].eq(expected).all(),
            f"Unexpected source metadata in {column}.")

expected_quarters = pd.period_range(spec["first_quarter"], spec["last_quarter"], freq="Q")
expected_lgas = sum(spec["metro_group_counts"].values())
require(panel["rental_area_name"].nunique() == expected_lgas, "Unexpected LGA count.")
require(len(panel) == expected_lgas * len(expected_quarters), "Unexpected row count.")
expected_dates = pd.PeriodIndex(panel["quarter"], freq="Q").to_timestamp(how="end").normalize()
actual_dates = pd.DatetimeIndex(
    panel["quarter_end"]
).as_unit("ns")

expected_dates = (
    pd.PeriodIndex(panel["quarter"], freq="Q")
    .to_timestamp(how="end")
    .normalize()
    .as_unit("ns")
)

require(
    np.array_equal(actual_dates.asi8, expected_dates.asi8),
    "Quarter labels and end dates disagree.",
)
for key, group in panel.groupby(["rental_area_name", "property_type"], sort=False):
    require(group.sort_values("quarter_end")["quarter"].tolist() == expected_quarters.astype(str).tolist(),
            f"Incomplete or changed quarterly coverage for {key}.")
for column in ["published_count", "median_weekly_rent"]:
    numeric = panel[column].to_numpy(dtype=float)
    require(np.isfinite(numeric).all() and (numeric > 0).all(), f"Invalid {column}.")
require(pd.api.types.is_integer_dtype(panel["published_count"]), "Counts must be integers.")
region_inventory = panel[["rental_area_name", "source_region"]].drop_duplicates()
require(len(region_inventory) == expected_lgas, "An LGA changes source group within the table.")
require(region_inventory.groupby("source_region").size().to_dict() == spec["metro_group_counts"],
        "Unexpected source-group composition.")

# Capture hashes for all existing upstream inputs. None will be written by this notebook.
tracked_paths = [INPUT, SPEC_PATH, QUALITY_PATH]
raw_path = ROOT / "data/raw/rent" / spec["filename"]
parquet_path = ROOT / "data/processed/rent_quarterly.parquet"
if raw_path.is_file():
    require(fingerprint(raw_path) == spec["sha256"], "The local raw workbook changed after its audit.")
    tracked_paths.append(raw_path)
if parquet_path.is_file():
    tracked_paths.append(parquet_path)
input_hashes = {path: fingerprint(path) for path in tracked_paths}
panel_before = panel.copy(deep=True)

print("INPUT CHECKPOINT PASSED")
print("Rows:", len(panel), "| Cleaned columns:", panel.shape[1])
print("LGA count:", expected_lgas)
print("Quarter coverage:", expected_quarters[0], "to", expected_quarters[-1])
print("Input CSV matches pipeline fingerprint: True")
display(panel.head(3))

INPUT CHECKPOINT PASSED
Rows: 3286 | Cleaned columns: 14
LGA count: 31
Quarter coverage: 1999Q2 to 2025Q3
Input CSV matches pipeline fingerprint: True


,rental_area_name,source_region,area_type,property_type,quarter,quarter_end,published_count,median_weekly_rent,source_file,source_sheet,source_reporting_quarter,source_excel_row,source_count_cell,source_median_cell
0,Banyule,North and West Metro,LGA,2 bedroom flats,1999Q2,1999-06-30,214,165.0,quarterly-median-rents-local-government-area-september-q...,2br Flat,2025Q3,57,C57,D57
1,Banyule,North and West Metro,LGA,2 bedroom flats,1999Q3,1999-09-30,202,165.0,quarterly-median-rents-local-government-area-september-q...,2br Flat,2025Q3,57,E57,F57
2,Banyule,North and West Metro,LGA,2 bedroom flats,1999Q4,1999-12-31,232,170.0,quarterly-median-rents-local-government-area-september-q...,2br Flat,2025Q3,57,G57,H57


## 3. Compare the latest published quarter

**Latest means latest in this snapshot**, not the current calendar quarter. One row below is one LGA's two-bedroom-flat statistic for September 2025.

We sort by median weekly rent and preserve ties. This is a comparison of published rents, **not a suitability ranking**. A low rent does not by itself establish good transport, enough available properties, or affordability relative to someone's income.

The summary below gives the **range of individual LGA medians**. It deliberately does not compute a metropolitan median by averaging or taking the median of these medians.

In [5]:
LATEST_QUARTER = spec["last_quarter"]
CATEGORY = spec["property_type"]
LATEST_LABEL = pd.Period(LATEST_QUARTER, freq="Q").strftime("%b %Y")
latest = panel.loc[panel["quarter"].eq(LATEST_QUARTER)].copy()
latest = latest.sort_values(["median_weekly_rent", "rental_area_name"], kind="stable").reset_index(drop=True)
require(len(latest) == expected_lgas, "Latest quarter must have one observation for every selected LGA.")
latest_columns = [
    "rental_area_name", "quarter", "median_weekly_rent", "published_count",
    "source_region", "source_count_cell", "source_median_cell",
]
display(latest[latest_columns].style.format({
    "median_weekly_rent": "${:,.0f}", "published_count": "{:,.0f}",
}))
low = float(latest["median_weekly_rent"].min())
high = float(latest["median_weekly_rent"].max())
print(f"Published quarter: {LATEST_QUARTER} ({LATEST_LABEL})")
print(f"Range across LGA medians: ${low:,.0f} to ${high:,.0f} per week")
print("Lowest-median LGA(s):", ", ".join(latest.loc[latest["median_weekly_rent"].eq(low), "rental_area_name"]))
print("Highest-median LGA(s):", ", ".join(latest.loc[latest["median_weekly_rent"].eq(high), "rental_area_name"]))

,rental_area_name,quarter,median_weekly_rent,published_count,source_region,source_count_cell,source_median_cell
0,Brimbank,2025Q3,$400,224,North and West Metro,HE58,HF58
1,Wyndham,2025Q3,$410,138,North and West Metro,HE69,HF69
2,Melton,2025Q3,$425,44,North and West Metro,HE64,HF64
3,Cardinia,2025Q3,$450,24,Southern Metro,HE81,HF81
4,Hume,2025Q3,$450,115,North and West Metro,HE61,HF61
5,Whittlesea,2025Q3,$460,117,North and West Metro,HE68,HF68
6,Greater Dandenong,2025Q3,$470,297,Southern Metro,HE85,HF85
7,Casey,2025Q3,$480,88,Southern Metro,HE82,HF82
8,Frankston,2025Q3,$490,145,Southern Metro,HE83,HF83
9,Hobsons Bay,2025Q3,$500,149,North and West Metro,HE60,HF60


Published quarter: 2025Q3 (Sep 2025)
Range across LGA medians: $400 to $730 per week
Lowest-median LGA(s): Brimbank
Highest-median LGA(s): Melbourne


## 4. Chart 1 — Latest LGA rental comparison

Each bar is an LGA; its length is the published median in AUD per week. Hover to see the observation count and source cell. The axis starts at zero so the bar lengths remain interpretable.

All charts will also be saved as self-contained HTML files in Section 12. The files work in a browser without a Plotly account or an online chart service.

In [6]:
figures = {}
DISPLAY_CHARTS = True  # Set False only when your notebook viewer cannot render Plotly.

def show_chart(figure):
    if DISPLAY_CHARTS:
        figure.show(renderer="plotly_mimetype")

fig = go.Figure(go.Bar(
    x=latest["median_weekly_rent"].tolist(),
    y=latest["rental_area_name"].tolist(),
    orientation="h",
    text=[f"${value:,.0f}" for value in latest["median_weekly_rent"]],
    textposition="outside", cliponaxis=False,
    customdata=latest[["published_count", "source_median_cell"]].to_numpy(),
    hovertemplate=("%{y}<br>Median: $%{x:,.0f}/week<br>Published count: %{customdata[0]:,.0f}"
                   "<br>Median source cell: %{customdata[1]}<extra></extra>"),
))
fig.update_layout(
    title=f"Published LGA median weekly rents — {LATEST_LABEL}<br><sup>{CATEGORY}; source metropolitan groups; not a recommendation</sup>",
    xaxis_title="Median weekly rent (AUD; nominal)", yaxis_title="LGA (source spelling)",
    height=1050, margin=dict(l=175, r=70, t=100, b=70),
    showlegend=False,
)
fig.update_xaxes(range=[0, high * 1.15], tickprefix="$")
fig.update_yaxes(categoryorder="array", categoryarray=latest["rental_area_name"].tolist(), autorange="reversed")
figures["01_latest_rents"] = fig
show_chart(fig)

## 5. Chart 2 — Follow selected LGAs over time

We initially use **Melbourne, Monash and Whitehorse**, the same examples already traced during the audit. They are illustrative, not a representative sample of all 31 LGAs. `Melbourne` here is the individual LGA.

Change `SELECTED_LGAS` only after the first complete run; use exact source spellings and keep the chart to a few lines. The data is sorted chronologically before drawing lines.

Values remain in nominal dollars. A change in an area's median can also reflect changes in which properties were newly let; this is not a matched-property rent-increase series. Do not add an explanation for a rise or fall without additional evidence.

In [7]:
SELECTED_LGAS = ["Melbourne", "Monash", "Whitehorse"]
require(1 <= len(SELECTED_LGAS) <= 6 and len(set(SELECTED_LGAS)) == len(SELECTED_LGAS),
        "Select one to six distinct LGA names for a readable history chart.")
unknown_lgas = set(SELECTED_LGAS) - set(panel["rental_area_name"])
require(not unknown_lgas, f"Unknown LGA names: {sorted(unknown_lgas)}")
history = panel.loc[panel["rental_area_name"].isin(SELECTED_LGAS)].copy()
history = history.sort_values(["rental_area_name", "quarter_end"], kind="stable")

fig = go.Figure()
for name in SELECTED_LGAS:
    area = history.loc[history["rental_area_name"].eq(name)].sort_values("quarter_end")
    fig.add_trace(go.Scatter(
        x=area["quarter_end"].tolist(), y=area["median_weekly_rent"].tolist(),
        name=name, mode="lines",
        customdata=area[["quarter", "published_count", "source_median_cell"]].to_numpy(),
        hovertemplate=("%{fullData.name}<br>%{customdata[0]}<br>Median: $%{y:,.0f}/week"
                       "<br>Published count: %{customdata[1]:,.0f}"
                       "<br>Median source cell: %{customdata[2]}<extra></extra>"),
    ))
fig.update_layout(
    title=f"Selected LGA rent histories — {CATEGORY}<br><sup>{spec['first_quarter']}–{LATEST_QUARTER}; nominal dollars; no causal attribution</sup>",
    xaxis_title="Quarter ending", yaxis_title="Median weekly rent (AUD; nominal)",
    height=550, hovermode="x unified", margin=dict(l=70, r=35, t=105, b=100),
    legend=dict(orientation="h", y=-0.2, x=0),
)
fig.update_yaxes(rangemode="tozero", tickprefix="$")
figures["02_selected_histories"] = fig
show_chart(fig)
print("Selected history rows:", len(history))

Selected history rows: 318


## 6. Calculate quarter-on-quarter and year-on-year changes

**Quarter-on-quarter (QoQ)** compares with the previous quarter. **Year-on-year (YoY)** compares with the same quarter one year earlier.

For example, for `2025Q3`:
- previous quarter = `2025Q2`;
- same quarter one year earlier = `2024Q3`.

For rent $R_t$ and the prior-year rent $R_{t-4}$:

$$\text{YoY change (AUD/week)} = R_t - R_{t-4}$$
$$\text{YoY change (\%)} = 100\left(\frac{R_t}{R_{t-4}}-1\right)$$

**Illustration only:** $600 to $630 is +$30/week and +5%. This is not a forecast or a statement about any particular LGA.

`groupby(...).shift(4)` moves observations **within each LGA and property category**. It means one year only because we first check that every group is sorted and has consecutive quarters. The function below refuses duplicate keys, gaps and non-positive rents rather than making a misleading comparison.

Technical reference: https://pandas.pydata.org/docs/reference/api/pandas.api.typing.SeriesGroupBy.shift.html

**Method distinction:** these are our percentage changes in published medians. DFFH's headline percentage changes use its stratified Rental Indices, not simply the median-rent changes calculated here. Source: https://www.dffh.vic.gov.au/publications/rental-report

In [8]:
def add_rent_changes(frame: pd.DataFrame) -> pd.DataFrame:
    """Return a new table with grouped lags; reject gaps before using row shifts."""
    required = set(KEY_COLUMNS) | {"median_weekly_rent", "published_count", "source_median_cell"}
    require(required.issubset(frame.columns), f"Missing columns: {sorted(required - set(frame.columns))}")
    require(not frame.empty, "Cannot analyse an empty table.")
    require(frame[KEY_COLUMNS].notna().all().all(), "Missing analytical key.")
    require(not frame.duplicated(KEY_COLUMNS).any(), "Duplicate analytical keys.")
    result = frame.copy(deep=True)
    periods = pd.PeriodIndex(result["quarter"], freq="Q")
    require(periods.astype(str).tolist() == result["quarter"].tolist(), "Quarter labels must use YYYYQn.")
    result["_quarter_order"] = periods.asi8
    groups = ["rental_area_name", "property_type"]
    result = result.sort_values(groups + ["_quarter_order"], kind="stable").reset_index(drop=True)
    rents = pd.to_numeric(result["median_weekly_rent"], errors="raise").to_numpy(dtype=float)
    require(np.isfinite(rents).all() and (rents > 0).all(), "Rents must be finite and positive for this comparison.")
    result["median_weekly_rent"] = rents
    for key, group in result.groupby(groups, sort=False):
        require(np.all(np.diff(group["_quarter_order"].to_numpy()) == 1),
                f"Quarter gap or duplicate period within {key}; do not interpret a row lag as one year.")
    grouped = result.groupby(groups, sort=False)
    for lag, suffix in [(1, "prev_quarter"), (4, "prev_year")]:
        result[f"rent_{suffix}"] = grouped["median_weekly_rent"].shift(lag)
        result[f"quarter_{suffix}"] = grouped["quarter"].shift(lag)
    result["count_prev_year"] = grouped["published_count"].shift(4)
    result["source_median_cell_prev_year"] = grouped["source_median_cell"].shift(4)
    result["qoq_change_aud"] = result["median_weekly_rent"] - result["rent_prev_quarter"]
    result["qoq_change_pct"] = 100 * (result["median_weekly_rent"] / result["rent_prev_quarter"] - 1)
    result["yoy_change_aud"] = result["median_weekly_rent"] - result["rent_prev_year"]
    result["yoy_change_pct"] = 100 * (result["median_weekly_rent"] / result["rent_prev_year"] - 1)
    return result.drop(columns="_quarter_order")

eda = add_rent_changes(panel)
expected_qoq = expected_lgas * (len(expected_quarters) - 1)
expected_yoy = expected_lgas * (len(expected_quarters) - 4)
require(int(eda["qoq_change_pct"].notna().sum()) == expected_qoq, "Unexpected QoQ coverage.")
require(int(eda["yoy_change_pct"].notna().sum()) == expected_yoy, "Unexpected YoY coverage.")
print("Rows with a valid QoQ comparison:", expected_qoq)
print("Rows with a valid YoY comparison:", expected_yoy)
print("Expected initial unavailable QoQ comparisons:", len(eda) - expected_qoq)
print("Expected initial unavailable YoY comparisons:", len(eda) - expected_yoy)
print("Earliest available YoY comparison:", str(expected_quarters[4]))
display(eda.loc[eda["rental_area_name"].eq(SELECTED_LGAS[0]), [
    "rental_area_name", "quarter", "median_weekly_rent", "quarter_prev_year",
    "rent_prev_year", "yoy_change_aud", "yoy_change_pct",
]].head(6))

Rows with a valid QoQ comparison: 3255
Rows with a valid YoY comparison: 3162
Expected initial unavailable QoQ comparisons: 31
Expected initial unavailable YoY comparisons: 124
Earliest available YoY comparison: 2000Q2


,rental_area_name,quarter,median_weekly_rent,quarter_prev_year,rent_prev_year,yoy_change_aud,yoy_change_pct
1802,Melbourne,1999Q2,290.0,<NA>,NaN,NaN,NaN
1803,Melbourne,1999Q3,280.0,<NA>,NaN,NaN,NaN
1804,Melbourne,1999Q4,300.0,<NA>,NaN,NaN,NaN
1805,Melbourne,2000Q1,300.0,<NA>,NaN,NaN,NaN
1806,Melbourne,2000Q2,310.0,1999Q2,290.0,20.0,6.896552
1807,Melbourne,2000Q3,310.0,1999Q3,280.0,30.0,10.714286


### Why the first four annual-change values are missing

Our source starts in `1999Q2`. The first four quarters in each LGA have no corresponding observation one year earlier in the file. Therefore, **31 × 4 = 124** annual-change values are unavailable, even though no original rents are missing. These are expected comparison gaps, not a cleaning failure.

Do not fill these with zero: zero would claim that rent stayed unchanged. The first available year-on-year comparison is `2000Q2` versus `1999Q2`.

## 7. Verify the calculations independently

The first check creates two **fictional** time series with known changes, then checks shuffling, duplicate keys and missing quarters. These values are not added to the official dataset.

The second check matches all actual observations by exact LGA, property category and quarter using a join. It confirms that the four-row lag selects exactly the same prior-year values as calendar-quarter matching.

In [9]:
fixture_rows = []
fixture_quarters = pd.period_range("2024Q3", periods=5, freq="Q").astype(str)
for name, values in [("Fixture A", [100, 105, 110, 115, 120]),
                     ("Fixture B", [500, 490, 480, 470, 450])]:
    for quarter, value in zip(fixture_quarters, values):
        fixture_rows.append({"rental_area_name": name, "quarter": quarter,
                             "property_type": "fictional test type", "median_weekly_rent": value,
                             "published_count": 20, "source_median_cell": "TEST_ONLY"})
fixture = pd.DataFrame(fixture_rows)
fixture_result = add_rent_changes(fixture)
known = fixture_result.loc[fixture_result["quarter"].eq("2025Q3")].set_index("rental_area_name")
np.testing.assert_allclose(known.loc[["Fixture A", "Fixture B"], "yoy_change_pct"], [20, -10])
np.testing.assert_allclose(known.loc[["Fixture A", "Fixture B"], "yoy_change_aud"], [20, -50])
pd.testing.assert_frame_equal(fixture_result, add_rent_changes(fixture.sample(frac=1, random_state=7)))
for damaged in [fixture.drop(index=2), pd.concat([fixture, fixture.iloc[[0]]], ignore_index=True)]:
    try:
        add_rent_changes(damaged)
    except ValueError:
        pass
    else:
        raise AssertionError("The change function accepted a quarter gap or duplicate key.")
print("Four calculation self-checks passed: known answers, row order, gap rejection, duplicate rejection.")

# Independent calendar-quarter matching, not another shift(4).
lookup = panel[KEY_COLUMNS + ["median_weekly_rent", "published_count", "source_median_cell"]].copy()
lookup["quarter"] = (pd.PeriodIndex(lookup["quarter"], freq="Q") + 4).astype(str)
lookup = lookup.rename(columns={"median_weekly_rent": "lookup_prior_rent",
                                "published_count": "lookup_prior_count",
                                "source_median_cell": "lookup_prior_cell"})
comparison = eda.merge(lookup, on=KEY_COLUMNS, how="left", validate="one_to_one")
np.testing.assert_allclose(comparison["rent_prev_year"].to_numpy(dtype=float),
                           comparison["lookup_prior_rent"].to_numpy(dtype=float),
                           rtol=0, atol=0, equal_nan=True)
np.testing.assert_allclose(comparison["count_prev_year"].to_numpy(dtype=float, na_value=np.nan),
                           comparison["lookup_prior_count"].to_numpy(dtype=float, na_value=np.nan),
                           rtol=0, atol=0, equal_nan=True)
require(comparison["source_median_cell_prev_year"].fillna("").tolist()
        == comparison["lookup_prior_cell"].fillna("").tolist(), "Prior-year cell reference mismatch.")
independent_matches = int(comparison["lookup_prior_rent"].notna().sum())
require(independent_matches == expected_yoy, "Incorrect number of independent prior-year matches.")
print("Independent calendar-quarter matches:", independent_matches)

Four calculation self-checks passed: known answers, row order, gap rejection, duplicate rejection.
Independent calendar-quarter matches: 3162


## 8. Chart 3 — Compare annual changes in September 2025

Keep **both rents** in the table so the denominator of the percentage is visible. A +$30/week difference from $600 is not the same percentage as a +$30/week difference from $400.

The source cells refer to the prior-year median and current median in the same worksheet. We retain the published counts for **both periods**. Unequal counts are context, not confidence intervals.

Sorting by change does not identify the best investment, predict future change or explain why an area's median moved. A negative value simply means the published median was lower than in the equivalent quarter of the previous year.

In [11]:
latest_changes = eda.loc[eda["quarter"].eq(LATEST_QUARTER)].copy()
latest_changes = latest_changes.sort_values(["yoy_change_pct", "rental_area_name"], kind="stable").reset_index(drop=True)
require(len(latest_changes) == expected_lgas and latest_changes["yoy_change_pct"].notna().all(),
        "The latest-quarter YoY comparison must cover all selected LGAs.")
growth_columns = [
    "rental_area_name", "quarter_prev_year", "rent_prev_year", "quarter", "median_weekly_rent",
    "yoy_change_aud", "yoy_change_pct", "count_prev_year", "published_count",
    "source_median_cell_prev_year", "source_median_cell",
]
display(latest_changes[growth_columns].style.format({
    "rent_prev_year": "${:,.0f}", "median_weekly_rent": "${:,.0f}",
    "yoy_change_aud": "{:+,.0f}", "yoy_change_pct": "{:+.2f}%",
    "count_prev_year": "{:,.0f}", "published_count": "{:,.0f}",
}))
fig = go.Figure(go.Bar(
    x=latest_changes["yoy_change_pct"].tolist(), y=latest_changes["rental_area_name"].tolist(),
    orientation="h", text=[f"{value:+.2f}%" for value in latest_changes["yoy_change_pct"]],
    textposition="outside", cliponaxis=False,
    customdata=latest_changes[["rent_prev_year", "median_weekly_rent", "yoy_change_aud",
                               "count_prev_year", "published_count"]].to_numpy(),
    hovertemplate=("%{y}<br>YoY change: %{x:+.2f}%<br>Previous-year median: $%{customdata[0]:,.0f}"
                   "<br>Current median: $%{customdata[1]:,.0f}<br>Difference: $%{customdata[2]:+,.0f}/week"
                   "<br>Prior count: %{customdata[3]:,.0f}; current count: %{customdata[4]:,.0f}<extra></extra>"),
))
fig.update_layout(
    title=f"Year-on-year change in LGA medians — {LATEST_LABEL}<br><sup>{CATEGORY}; not the official Rental Index or a forecast</sup>",
    xaxis_title="Change from the same quarter last year (%)", yaxis_title="LGA (source spelling)",
    height=1050, margin=dict(l=175, r=85, t=100, b=70), showlegend=False,
)
minimum_growth = float(latest_changes["yoy_change_pct"].min())
maximum_growth = float(latest_changes["yoy_change_pct"].max())
span = max(maximum_growth - minimum_growth, 1.0)
fig.update_xaxes(range=[min(0, minimum_growth) - 0.18 * span, max(0, maximum_growth) + 0.18 * span], ticksuffix="%")
fig.update_yaxes(categoryorder="array", categoryarray=latest_changes["rental_area_name"].tolist(), autorange="reversed")
fig.add_vline(x=0, line_width=1)
figures["03_latest_yoy_changes"] = fig
show_chart(fig)

,rental_area_name,quarter_prev_year,rent_prev_year,quarter,median_weekly_rent,yoy_change_aud,yoy_change_pct,count_prev_year,published_count,source_median_cell_prev_year,source_median_cell
0,Brimbank,2024Q3,$420,2025Q3,$400,-20,-4.76%,170,224,GX58,HF58
1,Hobsons Bay,2024Q3,$520,2025Q3,$500,-20,-3.85%,196,149,GX60,HF60
2,Wyndham,2024Q3,$420,2025Q3,$410,-10,-2.38%,146,138,GX69,HF69
3,Banyule,2024Q3,$545,2025Q3,$540,-5,-0.92%,421,282,GX57,HF57
4,Cardinia,2024Q3,$450,2025Q3,$450,+0,+0.00%,31,24,GX81,HF81
5,Merri-bek,2024Q3,$540,2025Q3,$540,+0,+0.00%,691,728,GX65,HF65
6,Maribyrnong,2024Q3,$530,2025Q3,$533,+3,+0.57%,465,402,GX62,HF62
7,Darebin,2024Q3,$525,2025Q3,$530,+5,+0.95%,520,473,GX59,HF59
8,Moonee Valley,2024Q3,$520,2025Q3,$525,+5,+0.96%,395,353,GX66,HF66
9,Casey,2024Q3,$470,2025Q3,$480,+10,+2.13%,93,88,GX82,HF82


## 9. Chart 4 — Keep published counts in view

A median accompanied by a small number of published observations deserves context. Counts alone do **not** give the rent distribution, a confidence interval, statistical significance or a current vacancy rate. We do not delete observations or impose an unsupported 'reliable/unreliable' threshold here.

This scatterplot displays each LGA's **published count** against its median rent for the same quarter. Hover over points to identify the areas. This is a descriptive view; no fitted trend line or causal interpretation is added.

In [12]:
count_context = latest_changes[[
    "rental_area_name", "quarter", "published_count", "count_prev_year",
    "median_weekly_rent", "yoy_change_pct", "source_count_cell", "source_median_cell",
]].sort_values(["published_count", "rental_area_name"], kind="stable").reset_index(drop=True)
print("Five smallest published counts in the latest-quarter subset:")
display(count_context.head(5))
print("Five largest published counts in the latest-quarter subset:")
display(count_context.tail(5))

fig = go.Figure(go.Scatter(
    x=count_context["published_count"].tolist(), y=count_context["median_weekly_rent"].tolist(),
    mode="markers", marker=dict(size=10),
    customdata=count_context[["rental_area_name", "yoy_change_pct", "source_count_cell"]].to_numpy(),
    hovertemplate=("%{customdata[0]}<br>Published count: %{x:,.0f}"
                   "<br>Median: $%{y:,.0f}/week<br>YoY median change: %{customdata[1]:+.2f}%"
                   "<br>Count source cell: %{customdata[2]}<extra></extra>"),
))
fig.update_layout(
    title=f"Published counts and median rents — {LATEST_LABEL}<br><sup>{CATEGORY}; counts are not vacancies or confidence intervals</sup>",
    xaxis_title="Published count for the quarter", yaxis_title="Median weekly rent (AUD; nominal)",
    height=570, margin=dict(l=70, r=40, t=105, b=70),
)
fig.update_xaxes(rangemode="tozero")
fig.update_yaxes(rangemode="tozero", tickprefix="$")
figures["04_counts_and_rents"] = fig
show_chart(fig)

Five smallest published counts in the latest-quarter subset:


,rental_area_name,quarter,published_count,count_prev_year,median_weekly_rent,yoy_change_pct,source_count_cell,source_median_cell
0,Nillumbik,2025Q3,10,12,515.0,5.532787,HE67,HF67
1,Cardinia,2025Q3,24,31,450.0,0.000000,HE81,HF81
2,Melton,2025Q3,44,60,425.0,6.250000,HE64,HF64
3,Yarra Ranges,2025Q3,48,40,530.0,10.416667,HE78,HF78
4,Knox,2025Q3,85,115,540.0,8.000000,HE73,HF73


Five largest published counts in the latest-quarter subset:


,rental_area_name,quarter,published_count,count_prev_year,median_weekly_rent,yoy_change_pct,source_count_cell,source_median_cell
26,Merri-bek,2025Q3,728,691,540.0,0.000000,HE65,HF65
27,Port Phillip,2025Q3,742,816,650.0,3.668262,HE88,HF88
28,Stonnington,2025Q3,751,862,675.0,4.651163,HE89,HF89
29,Glen Eira,2025Q3,760,780,600.0,2.214651,HE84,HF84
30,Melbourne,2025Q3,3209,3112,730.0,4.285714,HE63,HF63


## 10. Inspect the worked examples and write careful findings

This table returns to our three audit examples with their prior-year rents and changes. Verify one percentage yourself before accepting the result.

When writing a finding, include **area + property type + period + units**, then separate the observation from its explanation. 'The median increased by X%' is supported by this calculation. 'A transport upgrade caused the increase' is not supported by the rental table alone.

A useful structure is: **Observation → comparison → limitation.** The generated report will supply a factual summary, but your own interpretation belongs in a Markdown cell or your progress notes.

In [13]:
worked_examples = latest_changes.loc[latest_changes["rental_area_name"].isin(SELECTED_LGAS), growth_columns].copy()
worked_examples = worked_examples.sort_values("rental_area_name").reset_index(drop=True)
display(worked_examples.style.format({
    "rent_prev_year": "${:,.0f}", "median_weekly_rent": "${:,.0f}",
    "yoy_change_aud": "{:+,.0f}", "yoy_change_pct": "{:+.2f}%",
    "count_prev_year": "{:,.0f}", "published_count": "{:,.0f}",
}))
for row in worked_examples.itertuples(index=False):
    calculated = 100 * (row.median_weekly_rent / row.rent_prev_year - 1)
    require(np.isclose(calculated, row.yoy_change_pct), "Worked example does not match.")
    print(f"{row.rental_area_name}: 100 × ({row.median_weekly_rent:g} / {row.rent_prev_year:g} - 1) = {calculated:+.2f}%")

,rental_area_name,quarter_prev_year,rent_prev_year,quarter,median_weekly_rent,yoy_change_aud,yoy_change_pct,count_prev_year,published_count,source_median_cell_prev_year,source_median_cell
0,Melbourne,2024Q3,$700,2025Q3,$730,+30,+4.29%,"3,112","3,209",GX63,HF63
1,Monash,2024Q3,$570,2025Q3,$600,+30,+5.26%,442,421,GX76,HF76
2,Whitehorse,2024Q3,$580,2025Q3,$600,+20,+3.45%,532,464,GX77,HF77


Melbourne: 100 × (730 / 700 - 1) = +4.29%
Monash: 100 × (600 / 570 - 1) = +5.26%
Whitehorse: 100 × (600 / 580 - 1) = +3.45%


## 11. Important limits before exporting

**Different statistic:** our YoY and QoQ values use individual LGA medians. The publisher's headline changes use stratified Rental Indices to control for compositional shifts. Do not equate these statistics.

**Different properties can appear in different quarters:** these are new-letting statistics, not repeated measurements of identical properties. The publisher uses rental bond data as a proxy and excludes specified accommodation types, including student housing and rooms in dwellings.

**No metropolitan median from LGA medians:** even count-weighting the medians does not generally recover the median of all underlying rents. Use a published aggregate when that is needed.

**No inflation adjustment:** dollar changes describe nominal prices. We cannot call them real purchasing-power changes without a suitable price index.

**No full affordability or access assessment:** we have not added income, budget eligibility, transport, commute or environmental measures. Lower median rent is not a recommendation.

**No historical boundary reconstruction:** source spellings and groups are retained. Later geographic matching must document the boundary vintage and correspondence.

**Not a forecasting result:** the full snapshot is being explored descriptively. Future model evaluation must explicitly document what periods have already been inspected and set out a temporal evaluation protocol before tuning models.

Publisher's definitions: https://www.dffh.vic.gov.au/publications/rental-report

## 12. Export charts, tables and a generated findings report

This cell writes only derived EDA outputs. It does not alter `rent_quarterly.csv`, the Parquet copy, the source workbook, the pipeline's quality report, your registry or progress notes.

Each chart is a self-contained HTML file with Plotly's JavaScript embedded: open it locally in a browser, including offline. This avoids needing an image-export package. The files are larger than PNGs and stay under the project's already-ignored `data/interim/` folder.

`docs/rental_eda_summary.md` is **generated** and is replaced on reruns of this cell. Keep any personal notes in a separate Markdown cell or in `docs/week2_progress.md`.

Plotly export reference: https://plotly.com/python/interactive-html-export/

In [12]:
EDA_DIR = ROOT / "data/interim/rent_eda"
FIGURE_DIR = EDA_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_PATH = ROOT / "docs/rental_eda_summary.md"
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Check that even the in-memory base table has not been modified by the analysis.
pd.testing.assert_frame_equal(panel, panel_before, check_exact=True)
for path, before_hash in input_hashes.items():
    require(fingerprint(path) == before_hash, f"Upstream input changed during EDA: {path.name}")

tables = {
    "rental_panel_with_changes.csv": eda,
    "latest_lga_comparison.csv": latest_changes,
    "selected_lga_history.csv": eda.loc[eda["rental_area_name"].isin(SELECTED_LGAS)].copy(),
    "count_context.csv": count_context,
}
for filename, frame in tables.items():
    frame.to_csv(EDA_DIR / filename, index=False, date_format="%Y-%m-%d", encoding="utf-8")
for name, figure in figures.items():
    figure.write_html(FIGURE_DIR / f"{name}.html", include_plotlyjs=True,
                      full_html=True, auto_open=False, config={"responsive": True})

# Keep findings traceable to the original cells; preserve ties rather than picking an arbitrary winner.
def describe_extreme(frame, column, value):
    rows = frame.loc[np.isclose(frame[column].astype(float), value)].sort_values("rental_area_name")
    return "; ".join(
        f"{row.rental_area_name} (${row.median_weekly_rent:,.0f}/week; count {row.published_count:,}; "
        f"median cell {row.source_median_cell})"
        for row in rows.itertuples(index=False)
    )

positive = int(latest_changes["yoy_change_aud"].gt(0).sum())
negative = int(latest_changes["yoy_change_aud"].lt(0).sum())
unchanged = int(latest_changes["yoy_change_aud"].eq(0).sum())
prior_year_label = str(pd.Period(LATEST_QUARTER, freq="Q") - 4)
report_lines = [
    "# Rental EDA — First descriptive findings", "",
    "Generated by `notebooks/03_rental_eda.ipynb`; rerunning the export cell replaces this report.", "",
    "## Data and provenance", "",
    f"- Source file: `{spec['filename']}`; worksheet `{spec['sheet']}`.",
    f"- Property category: `{CATEGORY}`; source reporting snapshot: `{spec['reporting_quarter']}`.",
    f"- Scope: {expected_lgas} LGAs in the publisher's metropolitan groups; {len(expected_quarters)} quarters, {spec['first_quarter']}–{LATEST_QUARTER}.",
    f"- Cleaned observations: {len(panel):,}; input: `data/processed/rent_quarterly.csv`.",
    f"- CSV fingerprint: `{input_sha256}`.",
    f"- Publisher definitions: {spec['publisher_page']}", "",
    "## 1. Latest-quarter comparison", "",
    f"In {LATEST_QUARTER}, the published median weekly rents for {CATEGORY} across this LGA subset ranged from ${low:,.0f} to ${high:,.0f}.",
    f"Lowest observed LGA median(s): {describe_extreme(latest, 'median_weekly_rent', low)}.",
    f"Highest observed LGA median(s): {describe_extreme(latest, 'median_weekly_rent', high)}.",
    "These are extrema of individual LGA medians, not a metropolitan median or a suitability ranking.", "",
    "## 2. Selected histories", "",
    "The history chart uses " + ", ".join(SELECTED_LGAS) + ". They were selected as audit examples, not as a representative sample.",
    "Values are nominal AUD per week. Changes in the newly let dwelling mix can affect area medians; no causal explanation is assigned.", "",
    "## 3. Same-quarter annual changes", "",
    f"For {LATEST_QUARTER} versus {prior_year_label}, {positive} LGA medians rose, {negative} fell and {unchanged} were unchanged in the published values.",
    f"The minimum calculated YoY change was {minimum_growth:+.2f}%; the maximum was {maximum_growth:+.2f}%.",
    "This calculation is 100 × (current median / prior-year median − 1), not the publisher's stratified Rental Index.", "",
    "| LGA | Prior-year median (AUD/week) | Latest median (AUD/week) | Difference (AUD/week) | YoY change | Prior count | Current count | Prior / current median cells |",
    "|---|---:|---:|---:|---:|---:|---:|---|",
]
for row in worked_examples.itertuples(index=False):
    report_lines.append(
        f"| {row.rental_area_name} | {row.rent_prev_year:,.0f} | {row.median_weekly_rent:,.0f} | "
        f"{row.yoy_change_aud:+,.0f} | {row.yoy_change_pct:+.2f}% | {int(row.count_prev_year):,} | "
        f"{row.published_count:,} | {row.source_median_cell_prev_year} / {row.source_median_cell} |"
    )
report_lines += [
    "", "## 4. Published-count context", "",
    f"Latest-quarter counts in this subset range from {int(latest['published_count'].min()):,} to {int(latest['published_count'].max()):,}.",
    "Counts are retained without filtering. They do not establish vacancy rates, the rent distribution, confidence intervals or significance.", "",
    "## Checks completed", "",
    f"- {expected_qoq:,} QoQ comparisons and {expected_yoy:,} YoY comparisons.",
    f"- {len(eda) - expected_qoq} initial QoQ gaps and {len(eda) - expected_yoy} initial YoY gaps are expected; they were not filled with zero.",
    f"- All {independent_matches:,} prior-year rent, count and median-cell matches agree with independent calendar-quarter lookup.",
    "- Four synthetic self-checks passed: known answers, shuffled-input invariance, gap rejection and duplicate rejection.",
    "- Upstream input files and the in-memory base table were unchanged.", "",
    "## Limits", "",
    "New-letting medians can change with dwelling composition. These are nominal prices, not inflation-adjusted values. "
    "The source excludes certain accommodation types. Do not aggregate the LGA medians into a purported metropolitan median. "
    "No household income, transport, commute, boundaries or environment are included in this analysis. "
    "No forecasts, individual-property recommendations, vacancy estimates or causal effects have been established.", "",
    "## Outputs", "",
    "Tables: `data/interim/rent_eda/`. Interactive charts: `data/interim/rent_eda/figures/`.",
    "Every latest-quarter comparison retains source metadata and the prior-year median cell. "
    "The plotted historical values can be traced using the exported selected-history table.", "",
    "## Technical references", "",
    "- https://pandas.pydata.org/docs/reference/api/pandas.api.typing.SeriesGroupBy.shift.html",
    "- https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html",
    "- https://plotly.com/python/interactive-html-export/", "",
]
REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")

# Verify the exported analysis values before declaring success.
reloaded_eda = pd.read_csv(EDA_DIR / "rental_panel_with_changes.csv")
require(len(reloaded_eda) == len(eda), "Exported EDA table row count changed.")
require(reloaded_eda[KEY_COLUMNS].astype(str).values.tolist() == eda[KEY_COLUMNS].astype(str).values.tolist(),
        "Exported EDA keys changed.")
for column in ["median_weekly_rent", "rent_prev_quarter", "rent_prev_year",
               "qoq_change_aud", "qoq_change_pct", "yoy_change_aud", "yoy_change_pct"]:
    np.testing.assert_allclose(reloaded_eda[column].to_numpy(dtype=float), eda[column].to_numpy(dtype=float),
                               rtol=1e-12, atol=1e-12, equal_nan=True)
for name in figures:
    chart_file = FIGURE_DIR / f"{name}.html"
    require(chart_file.is_file() and "Plotly.newPlot" in chart_file.read_text(encoding="utf-8"),
            f"Missing or invalid chart export: {name}")
for path, before_hash in input_hashes.items():
    require(fingerprint(path) == before_hash, f"Upstream file changed during export: {path.name}")

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "builder": "notebooks/03_rental_eda.ipynb",
    "pandas_version": pd.__version__, "plotly_version": plotly.__version__,
    "source_snapshot": spec["reporting_quarter"], "source_sha256": spec["sha256"],
    "input_csv_sha256": input_sha256, "cleaned_rows": len(panel), "lga_count": expected_lgas,
    "quarter_count": len(expected_quarters), "latest_quarter": LATEST_QUARTER,
    "property_type": CATEGORY, "selected_history_lgas": SELECTED_LGAS,
    "qoq_comparisons": expected_qoq, "yoy_comparisons": expected_yoy,
    "expected_initial_missing_qoq": len(eda) - expected_qoq,
    "expected_initial_missing_yoy": len(eda) - expected_yoy,
    "independent_prior_year_matches": independent_matches,
    "synthetic_self_checks": 4, "upstream_inputs_unchanged": True,
    "output_files": {
        str(path.relative_to(ROOT)): fingerprint(path)
        for path in [*[EDA_DIR / name for name in tables],
                     *[FIGURE_DIR / f"{name}.html" for name in figures], REPORT_PATH]
    },
}
(EDA_DIR / "eda_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print("EDA CHECKPOINT PASSED")
print("Cleaned input rows:", len(panel))
print("Latest-quarter comparison rows:", len(latest_changes))
print("Latest quarter:", LATEST_QUARTER)
print("Valid QoQ comparisons:", expected_qoq)
print("Valid YoY comparisons:", expected_yoy)
print("Expected initial unavailable YoY comparisons:", len(eda) - expected_yoy)
print("Independent prior-year matches:", independent_matches)
print("Calculation self-checks: 4 passed")
print("Interactive charts saved:", len(figures))
print("EDA tables saved:", len(tables))
print("Input files unchanged: True")
print("Saved report:", REPORT_PATH.relative_to(ROOT))
print("Saved EDA outputs:", EDA_DIR.relative_to(ROOT))

EDA CHECKPOINT PASSED
Cleaned input rows: 3286
Latest-quarter comparison rows: 31
Latest quarter: 2025Q3
Valid QoQ comparisons: 3255
Valid YoY comparisons: 3162
Expected initial unavailable YoY comparisons: 124
Independent prior-year matches: 3162
Calculation self-checks: 4 passed
Interactive charts saved: 4
EDA tables saved: 4
Input files unchanged: True
Saved report: docs\rental_eda_summary.md
Saved EDA outputs: data\interim\rent_eda


## Your checkpoint and interpretation exercise

Save this notebook with **Ctrl+S**. Open the generated findings report and at least one exported HTML chart. The final checkpoint confirms that the files were created; your visual inspection checks that their labels and layout are readable.

Explain these in your own words:

1. Why are there 124 unavailable annual changes when the cleaned rental table has no missing rent values?
2. What is the difference between a dollar change and a percentage change?
3. Why is an area's low median rent not sufficient to recommend that area?
4. Why must we not call this calculation the official Rental Index?

Write one finding with the **LGA, property category, period, value/comparison and one limitation**. Keep it factual; do not invent a cause.

Send the final `EDA CHECKPOINT PASSED` output, or the first error. Keep the existing notebooks and pipeline files. No Streamlit changes, forecast training or new dataset downloads are required for this checkpoint.

The next step after reviewing the charts is to preserve the EDA milestone and begin the geographic-boundary matching audit.